# 03 — Churn Model

This notebook trains the **churn** model, evaluates it with cross-validation and inspects coefficients

## Modeling choices

- **Prediction target:** 90-day churn risk for tenants active on 2024-06-30
- **Model family:** logistic regression
- **Validation:** 5-fold stratified cross-validation
- **Leakage rule:** all features must be available on or before the snapshot date

In [13]:

from pathlib import Path
import sys
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd().resolve().parents[0]
sys.path.append(str(PROJECT_ROOT / "src"))

import syskit_utils as su

DB_PATH = "saas_dataset.sqlite"
TABLES = su.load_tables(DB_PATH)



In [14]:
def get_model_spec() -> tuple:
    numeric_features = [
        "arr",
        "tenure_days",
        "users_total",
        "event_types_used_30d",
        "value_share_30d",
        "trend45_total_events",
        "trend45_active_users",
        "positive_touch_rate_90d",
        "negative_touch_rate_90d",
        "no_response_touch_rate_90d",
        "trend45_active_users_value",
        "trend45_value_events",
        "days_to_renewal_<180",
        "expansion_signal",
        "health_score_0_100",

    ]
    categorical_features = [
    ]
    return numeric_features, categorical_features

def fit_logistic_cv(feature_store: pd.DataFrame, c_value: float = 0.1, random_state: int = 42) -> dict:
    model_df = feature_store.loc[feature_store["snapshot_active"] == 1].copy()
    y = model_df["churn_90d"].astype(int)

    numeric_features, categorical_features = get_model_spec()
    X = model_df[numeric_features + categorical_features].copy()

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_features,
            ),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                categorical_features,
            ),
        ]
    )

    model = LogisticRegression(max_iter=5000, C=c_value, solver="liblinear")
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
    oof_prob = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba")[:, 1]

    k = max(1, int(np.ceil(len(oof_prob) * 0.10)))
    top_idx = np.argsort(-oof_prob)[:k]
    metrics = {
        "snapshot_active_tenants": int(len(model_df)),
        "positive_90d_churners": int(y.sum()),
        "roc_auc": float(roc_auc_score(y, oof_prob)),
        "precision_at_top_decile": float(y.iloc[top_idx].mean()),
        "recall_at_top_decile": float(y.iloc[top_idx].sum() / y.sum()),
    }

    pipe.fit(X, y)
    feature_names = pipe.named_steps["preprocessor"].get_feature_names_out()
    coef_df = pd.DataFrame(
        {"feature": feature_names, "coefficient": pipe.named_steps["model"].coef_[0]}
    ).sort_values("coefficient")

    model_df["churn_risk_90d"] = oof_prob
    model_df["risk_rank"] = model_df["churn_risk_90d"].rank(ascending=False, method="first")
    return {
        "metrics": metrics,
        "model_df": model_df,
        "pipeline": pipe,
        "coefficients": coef_df,
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
    }



In [15]:

feature_store = pd.read_csv(PROJECT_ROOT / "results" / "tenant_feature_store_snapshot_2024_06_30.csv", parse_dates=[
    "contract_start_date","renewal_date","churn_date","created_at","last_crm_touch_date"
])

fit = fit_logistic_cv(feature_store, c_value=0.1)
scored = fit["model_df"]
coefficients = fit["coefficients"]

pd.Series(fit["metrics"]).to_frame("value")


,value
snapshot_active_tenants,450.000000
positive_90d_churners,23.000000
roc_auc,0.931066
precision_at_top_decile,0.400000
recall_at_top_decile,0.782609


## Why these metrics

Accuracy is not informative here because only a minority of active tenants churn within 90 days. The most useful metrics are:

- **Top-decile precision** measures how many tenants in the highest-risk 10% actually churn. This directly reflects the quality of the outreach list that CS would prioritize first.

- **Top-decile recall** measures how many of all true churners are captured inside that highest-risk 10%. This reflects coverage: how much churn risk we catch with limited intervention bandwidth.

In [16]:
display(coefficients.sort_values("coefficient", ascending=False).head(5))
display(coefficients.sort_values("coefficient", ascending=True).head(5))

,feature,coefficient
9,num__no_response_touch_rate_90d,0.267549
8,num__negative_touch_rate_90d,0.242537
2,num__users_total,0.137204
12,num__days_to_renewal_<180,0.136513
0,num__arr,0.088666


,feature,coefficient
1,num__tenure_days,-0.476047
7,num__positive_touch_rate_90d,-0.183616
4,num__value_share_30d,-0.167670
3,num__event_types_used_30d,-0.088529
14,num__health_score_0_100,-0.071914


## Interpreting the coefficients

Coefficients are shown on the **standardized feature scale** for numeric inputs.

- A **positive** coefficient means higher values of that feature are associated with **higher churn risk**.
- A **negative** coefficient means higher values are associated with **lower churn risk**.

Based on the coefficient tables from this run:

**Features most associated with higher churn risk (top positive coefficients):**
- `no_response_touch_rate_90d`
- `negative_touch_rate_90d`
- `users_total`
- `days_to_renewal_<180`
- `arr`

**Features most associated with lower churn risk (top negative coefficients):**
- `tenure_days`
- `positive_touch_rate_90d`
- `value_share_30d`
- `event_types_used_30d`
- `health_score_0_100`

This suggests that weak CRM engagement quality (no response / negative outcomes) and near-term renewal pressure are risk signals. Also larger companies (more users, larger arr) have higher churn risk.

Longer tenure, healthier product-value usage patterns, and positive CRM interactions are protective.



In [17]:
scored.to_csv(PROJECT_ROOT / "results" / "tenant_scores_snapshot_2024_06_30.csv", index=False)

## Limitations

1. The dataset is small, with only 23 churners which means model performance should be closelly monitored with future data.
2. Model is trained/evaluated on tenants active at one snapshot date (2024-06-30), therefore model might not be generalizable on future periods.
3. No external holdout test set. Due to small dataset I used out-of-fold predictions for validation, but there is no final untouched dataset for confirmation.
4. The model predicts whether a customer will churn within the next 90 days. This horizon may be too long, since earlier analysis showed that meaningful behavior changes typically begin about 45 days before churn.